In [0]:
import requests
import pandas as pd
import time

BASE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
HEADERS = {"Accept": "application/json"}

def fetch_complaints(date_min, date_max, target_rows, page_size=100):
    all_rows = []
    frm = 0

    while len(all_rows) < target_rows:
        params = {
            "size": page_size,
            "frm": frm,
            "date_received_min": date_min,
            "date_received_max": date_max,
            "sort": "created_date_desc",
        }
        response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=15)
        response.raise_for_status()
        data = response.json()

        hits = data.get("hits", {}).get("hits", [])
        if not hits:
            break

        all_rows.extend([h["_source"] for h in hits])
        print(f"  fetched {len(all_rows)} / {target_rows} rows so far...")
        frm += page_size
        time.sleep(0.3)

    return pd.DataFrame(all_rows[:target_rows])

print("Pulling full load...")
full_load = fetch_complaints(date_min="2026-06-01", date_max="2026-08-31", target_rows=1000)
full_load.to_csv("full_load_sample.csv", index=False)
print(f"Full load saved: {len(full_load)} rows\n")

print("Pulling incremental load...")
incremental_load = fetch_complaints(date_min="2026-09-01", date_max="2026-09-07", target_rows=200)
incremental_load.to_csv("incremental_load_sample.csv", index=False)
print(f"Incremental load saved: {len(incremental_load)} rows")

In [0]:
import shutil, os

repo_path = "/Workspace/Users/ssalmanaali80@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples"
os.makedirs(repo_path, exist_ok=True)

shutil.copy("full_load_sample.csv", f"{repo_path}/full_load_sample.csv")
shutil.copy("incremental_load_sample.csv", f"{repo_path}/incremental_load_sample.csv")